In [ ]:
!pip install -U langchain==0.3.24 langgraph==0.3.33 langchain-tavily==0.1.6 langgraph-supervisor langchain-google-genai

In [ ]:
import os
os.environ["LANGCHAIN_API_KEY"] = "###"
os.environ["LANGCHAIN_PROJECT"] = "Visual-Agentic-AI"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["TAVILY_API_KEY"] = "###"
os.environ["GOOGLE_API_KEY"] = "#####"

# **Visual Agentic Agent**

In [ ]:
!pip install -U langchain-google-genai google-generativeai google-genai

In [46]:
# Keep this cell focused on non-LangChain extras to avoid dependency conflicts.
!pip install --upgrade -qq arxiv duckduckgo-search wikipedia python-magic ultralytics

In [149]:
import base64
from typing import Annotated, List, Union

import magic
import requests
from IPython.display import Image, display
from langchain_community.tools import (
    ArxivQueryRun,
    DuckDuckGoSearchResults,
    WikipediaQueryRun,
)
from langchain_community.utilities import (
    ArxivAPIWrapper,
    DuckDuckGoSearchAPIWrapper,
    WikipediaAPIWrapper,
)
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, create_react_agent, tools_condition
from pydantic import BaseModel, Field
from typing_extensions import TypedDict


from typing import Optional

from langchain_core.callbacks import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)
from langchain_core.tools import BaseTool
from langchain_core.tools import tool
from langchain_core.tools.base import ArgsSchema
from pydantic import BaseModel, Field

from ultralytics import YOLO

## **LLMs**

In [150]:
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

## **Research Agent**

### **Tool**

In [151]:
from langchain_community.tools import (
    ArxivQueryRun,
    DuckDuckGoSearchResults,
    WikipediaQueryRun,
)
from langchain_community.utilities import (
    ArxivAPIWrapper,
    DuckDuckGoSearchAPIWrapper,
    WikipediaAPIWrapper,
)

In [ ]:
arxiv_wrapper = ArxivAPIWrapper(
    top_k_results=2, doc_content_chars_max=1000
)
arxiv = ArxivQueryRun(
    api_wrapper=arxiv_wrapper,
    description="Search for papers on a given topic using Arxiv"
)
arxiv.invoke("Convolutional Neural Networks")

In [ ]:
wikipedia_wrapper = WikipediaAPIWrapper()
wikipedia = WikipediaQueryRun(
    api_wrapper=wikipedia_wrapper,
    description="Search for information on a given topic using Wikipedia"
)
wikipedia.invoke("machine learning")

### **Agent**

In [154]:
from langchain_core.messages import convert_to_messages

def pretty_print_message(message, indent=False):
    pretty_message = message.pretty_repr(html=True)
    if not indent:
        print(pretty_message)
        return

    indented = "\n".join("\t" + c for c in pretty_message.split("\n"))
    print(indented)

def pretty_print_messages(update, last_message=False):
    is_subgraph = False
    if isinstance(update, tuple):
        ns, update = update
        # skip parent graph updates in the printouts
        if len(ns) == 0:
            return

        graph_id = ns[-1].split(":")[0]
        print(f"Update from subgraph {graph_id}:")
        print("\n")
        is_subgraph = True

    for node_name, node_update in update.items():
        update_label = f"Update from node {node_name}:"
        if is_subgraph:
            update_label = "\t" + update_label

        print(update_label)
        print("\n")

        messages = convert_to_messages(node_update["messages"])
        if last_message:
            messages = messages[-1:]

        for m in messages:
            pretty_print_message(m, indent=is_subgraph)
        print("\n")

In [155]:
from langgraph.prebuilt import create_react_agent

research_agent = create_react_agent(
    model=llm,
    tools=[arxiv, wikipedia],
    prompt=(
        "You are a research agent.\n\n"
        "INSTRUCTIONS:\n"
        "- Assist ONLY with research-related tasks.\n"
        "- Use tools when needed and prioritize factual, verifiable information.\n"
        "- After finishing, respond directly with your final research result.\n"
        "- Keep the answer concise and focused on findings.\n"
    ),
    name="research_agent",
)

In [ ]:
for chunk in research_agent.stream(
    {"messages": [{"role": "user", "content": "Attention mechanisms in deep learning"}]}
):
    pretty_print_messages(chunk)

## **Vision Agent**

In [156]:
import os
import magic
import requests
import base64

def encode_image(image_path_or_url: str, get_mime_type: bool = False):
    if image_path_or_url.startswith("http"):
        try:
            response = requests.get(image_path_or_url, stream=True)
            response.raise_for_status()
            image = response.content
            mime_type = response.headers.get("content-type", None)
            base64_encoded = base64.b64encode(image).decode('utf-8')
            if get_mime_type:
                return base64_encoded, mime_type
            else:
                return base64_encoded
        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
            if get_mime_type:
                return None, None
            else:
                return None
    else:
        if not os.path.exists(image_path_or_url):
            return None, None
        mime_type = magic.Magic(mime=True).from_file(image_path_or_url)
        if mime_type.startswith("image/"):
            with open(image_path_or_url, "rb") as image_file:
                if get_mime_type:
                    return base64.b64encode(image_file.read()).decode("utf-8"), mime_type
                else:
                    return base64.b64encode(image_file.read()).decode("utf-8")
        else:
            if get_mime_type:
                return None, None
            else:
                return None

### **URL Extractor**

In [157]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

class ImageInput(BaseModel):
    image_path_or_url: str = Field(description="Image path or URL")

parser = PydanticOutputParser(pydantic_object=ImageInput)

prompt = PromptTemplate.from_template(
    "Extract the image path or URL from the following input:\n\n{input}\n\n{format_instructions}"
).partial(format_instructions=parser.get_format_instructions())

extractor_chain = prompt | llm | parser


In [ ]:
output = extractor_chain.invoke({"input": 'image_path: "https://example.com/image.png"'})
output

### **Image Describer Tool**

In [158]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage

class ImageDescription(BaseModel):
    image_description: str = Field(description="Detailed description of the image")

def image_describer_prompt_func(inputs: dict):
    image_path_or_url = inputs["image_path_or_url"]
    image_b64, image_mime_type = encode_image(image_path_or_url, get_mime_type=True)

    image_describer_chat_template = ChatPromptTemplate.from_messages([
        SystemMessage(
            content="""You are an expert image describer. When presented with an image, provide a detailed, accurate, and objective description of its visible content. Focus on aspects such as:
            - Objects present, their positions, and relationships
            - Colors, lighting, composition, and textures
            - Actions or dynamics, if any (e.g., people walking, water flowing)
            - Contextual or inferred information (e.g., likely setting, era, or activity)

            Avoid adding information that is not visible or cannot be reasonably inferred from the image. Do not speculate or inject personal opinion unless explicitly requested. If text appears in the image, transcribe it accurately."""),
        HumanMessage(content=[
            {"type": "text", "text": "Describe the following image for me:"},
            {
                "type": "image_url",
                "image_url": {"url": f"data:{image_mime_type};base64,{image_b64}", "detail": "low"}
            }
        ])
    ])
    return image_describer_chat_template.invoke({})

In [159]:
image_describer_agent = image_describer_prompt_func | llm.with_structured_output(ImageDescription)


In [ ]:
image_describer_agent.invoke({
    "image_path_or_url": "https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png"}
    )

In [160]:
from langchain_core.tools import BaseTool
from typing import Optional
from langchain_core.tools.base import ArgsSchema
from langchain_core.callbacks import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)

class ImageDescriberInput(BaseModel):
    text: str = Field(description="Path or URL to the image in the format PNG or JPG/JPEG")

class ImageDescriberTool(BaseTool):
    name: str = "image_describer"
    description: str = "This tool can describe the image in a detailed way"
    args_schema: Optional[ArgsSchema] = ImageDescriberInput
    return_direct: bool = True

    def _run(self, text: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """Use the tool."""
        try:
            parsed: ImageInput = extractor_chain.invoke({"input": text})
        except Exception as e:
            return f"Failed to extract image URL: {str(e)}"

        image_path_or_url = parsed.image_path_or_url
        if not image_path_or_url:
            return "No image URL found in the input."
        output = image_describer_agent.invoke({"image_path_or_url": image_path_or_url})
        return output.image_description

    async def _arun(self, image_path_or_url: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> str:
        """Use the tool asynchronously."""
        return self._run(image_path_or_url, run_manager=run_manager)

In [161]:
image_describer_tool = ImageDescriberTool()


In [ ]:
image_describer_tool.invoke("https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png")

### **Object Detection Tool**

In [162]:
from ultralytics import YOLO

yolo_model = YOLO("yolo11x.pt")

In [163]:
from typing import Annotated
from langchain_core.tools import tool

class ObjectDetectingAndCountingInput(BaseModel):
    text: str = Field(description="Path or URL to the image in the format PNG or JPG/JPEG")

@tool(
    "detect_and_count_objects",
    description="Detect and count objects within the image. The return will be a dictionary, containing the counting dictionary (counting how many instance of each object class) and a list of dictionaries, containing the object names, confidence scores, and location in the image (in (x1, x2, y1, y2) format).",
    args_schema=ObjectDetectingAndCountingInput
)
def detect_and_count_object_tool(
    text: Annotated[str, "Path or URL to the image"]
):
    """Detect objects within the image using YOLOv11 model"""

    try:
        parsed: ImageInput = extractor_chain.invoke({"input": text})
    except Exception as e:
        return f"Failed to extract image URL: {str(e)}"

    image_path_or_url = parsed.image_path_or_url
    if not image_path_or_url:
        return "No image URL found in the input."

    results = yolo_model(image_path_or_url, verbose=False)

    detections = []
    counting = {}

    # Process each result
    for result in results:
        boxes = result.boxes
        class_names = result.names

        for box in boxes:
            class_id = int(box.cls[0])
            class_name = class_names[class_id]
            confidence = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            detections.append({
                'class': class_name,
                'confidence': confidence,
                'bbox': (x1, y1, x2, y2)
            })

            counting[class_name] = counting.get(class_name, 0) + 1

    return str({'counting': counting, 'detections': detections})

In [ ]:
detect_and_count_object_tool.invoke("https://s3.amazonaws.com/cdn-origin-etr.akc.org/wp-content/uploads/2018/04/24144817/American-Staffordshire-Terrier-lying-outdoors-next-to-a-kitten-that-is-playing-with-the-dogs-nose.jpg")

### **Agent**

In [164]:
vision_agent = create_react_agent(
    model = llm,
    tools = [image_describer_tool, detect_and_count_object_tool],
    prompt = (
        "You are a vision agent.\n\n"
        "INSTRUCTIONS:\n"
        "- Assist ONLY with visual tasks (e.g.,describing images, detecting and counting objects)\n"
        "- Use only the tools provided to analyze visualinputs\n"
        "- After completing your task, respond to thesupervisor directly\n"
        "- Respond ONLY with the results of your work, doNOT include ANY other text."
    ),
    name = "vision_agent"
)
    

In [ ]:
for chunk in vision_agent.stream(
    {"messages": [{"role": "user", "content": "how many dog in image: https://s3.amazonaws.com/cdn-origin-etr.akc.org/wp-content/uploads/2018/04/24144817/American-Staffordshire-Terrier-lying-outdoors-next-to-a-kitten-that-is-playing-with-the-dogs-nose.jpg"}]}
):
    pretty_print_messages(chunk)

## **Critic Agent**

In [165]:
from langgraph.prebuilt import create_react_agent

critic_agent = create_react_agent(
    model=llm,
    tools=[],
    prompt=(
        "You are a critic and synthesis agent.\n\n"
        "GOALS:\n"
        "- Verify consistency and factual alignment between outputs from other agents.\n"
        "- Merge partial findings into one concise final answer for the user.\n"
        "- Explicitly call out uncertainty and missing evidence when needed.\n\n"
        "OUTPUT FORMAT (strict):\n"
        "FINAL_ANSWER: <clear final answer>\n"
        "EVIDENCE: <short bullets or compact paragraph based on available outputs>\n"
        "CONFIDENCE: <high|medium|low with one-line reason>\n"
        "LIMITATIONS: <what could not be verified or may be incomplete>"
    ),
    name="critic_agent",
)

## **Supervisor Agent (3 agents)**

In [166]:
from langgraph_supervisor import create_supervisor

supervisor = create_supervisor(
    model=llm,
    agents=[research_agent, vision_agent, critic_agent],
    prompt=(
        "You are a supervisor coordinating three agents: research_agent, vision_agent, critic_agent.\n\n"
        "ROUTING POLICY:\n"
        "- Use research_agent for research, background theory, and paper discovery.\n"
        "- Use vision_agent for image understanding, object detection/counting, and visual attributes.\n"
        "- Use critic_agent to synthesize and validate before final answer delivery.\n\n"
        "EXECUTION RULES:\n"
        "- Delegate to only one agent at a time.\n"
        "- Do not call agents in parallel.\n"
        "- Do not do domain work yourself; only orchestrate.\n"
        "- For mixed requests (research + vision), call needed specialist agents first, then always call critic_agent last.\n"
        "- Before final response, ensure critic_agent has produced the required format: FINAL_ANSWER, EVIDENCE, CONFIDENCE, LIMITATIONS."
    ),
    add_handoff_back_messages=True,
    output_mode="full_history",
).compile()

## **Inference**

In [ ]:
for chunk in supervisor.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the concept visualized in the image? Image: https://huggingface.co/datasets/tmnam20/Storage/resolve/main/rope.png Provide me detailed information about the concept. If possible, give me some research papers about it.",
            }
        ]
    },
):
    pretty_print_messages(chunk, last_message=True)

final_message_history = chunk["supervisor"]["messages"]